# Load the dataset and labels

In [1]:
import torch
from torchvision import datasets, transforms

# Load the training dataset
train_dataset = datasets.Food101(
    root="./data",
    split="train",
    download=True
)

# Load the test dataset
test_dataset = datasets.Food101(
    root="./data",
    split="test",
    download=True
)

labels = train_dataset.classes

print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))
print(train_dataset[0][0])  # Print the first training image
print(train_dataset[0][1])  # Print the label of the first training image

print(labels)

Training images: 75750
Test images: 25250
<PIL.Image.Image image mode=RGB size=512x512 at 0x1C646D7FD10>
23
['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare', 'beet_salad', 'beignets', 'bibimbap', 'bread_pudding', 'breakfast_burrito', 'bruschetta', 'caesar_salad', 'cannoli', 'caprese_salad', 'carrot_cake', 'ceviche', 'cheese_plate', 'cheesecake', 'chicken_curry', 'chicken_quesadilla', 'chicken_wings', 'chocolate_cake', 'chocolate_mousse', 'churros', 'clam_chowder', 'club_sandwich', 'crab_cakes', 'creme_brulee', 'croque_madame', 'cup_cakes', 'deviled_eggs', 'donuts', 'dumplings', 'edamame', 'eggs_benedict', 'escargots', 'falafel', 'filet_mignon', 'fish_and_chips', 'foie_gras', 'french_fries', 'french_onion_soup', 'french_toast', 'fried_calamari', 'fried_rice', 'frozen_yogurt', 'garlic_bread', 'gnocchi', 'greek_salad', 'grilled_cheese_sandwich', 'grilled_salmon', 'guacamole', 'gyoza', 'hamburger', 'hot_and_sour_soup', 'hot_dog', 'huevos_rancheros', 'hummus', 'ic

# Download the ViT

In [2]:
from transformers import ViTForImageClassification, AutoImageProcessor

model_id = "google/vit-base-patch16-224-in21k"
vit_model = ViTForImageClassification.from_pretrained(model_id, num_labels=101)
vit_processor = AutoImageProcessor.from_pretrained(model_id, backend="torchvision")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Implement the data collator

In [3]:
def vit_collate_fn(batch):
    images = [example[0] for example in batch]
    labels = [example[1] for example in batch]
    inputs = vit_processor(images, return_tensors="pt", do_convert_rgb=True)
    inputs["labels"] = torch.tensor(labels)
    return inputs

# Train the ViT model and compute accuracy

In [4]:
from transformers import Trainer, TrainingArguments

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    accuracy = (predictions == torch.tensor(labels)).float().mean().item()
    return {"accuracy": accuracy}

args = TrainingArguments("my_food101_vit", per_device_train_batch_size=32, 
                         eval_strategy="epoch", num_train_epochs=3,
                         remove_unused_columns=False, save_strategy="epoch", bf16=True)
trainer = Trainer(model=vit_model, args=args, data_collator=vit_collate_fn, 
                  train_dataset=train_dataset, eval_dataset=test_dataset,
                  compute_metrics=compute_metrics)
train_output = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.158702,0.737514,0.861941
2,0.451797,0.459921,0.887049
3,0.225417,0.423978,0.893624


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# Classification using CLIP, Zero Shot

## Load the model

In [5]:
from transformers import pipeline

model_id = "openai/clip-vit-base-patch32"
clip_pipeline = pipeline(task="zero-shot-image-classification", model=model_id, device_map="auto", dtype="auto")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## Evaluate on the test data and compute accuracy

In [6]:
from pathlib import Path

test_file_names = [line.strip() for line in Path("data/food-101/meta/test.txt").read_text(encoding="utf-8").splitlines() if line.strip()]
test_file_names = test_file_names
#print(test_file_names)

paths = [f"data/food-101/images/{n}.jpg" for n in test_file_names]  # names are already "class/id"
truths = [n.split("/")[0] for n in test_file_names]

results = clip_pipeline(paths, candidate_labels=labels,
                        hypothesis_template="This is a photo of a {}.",
                        batch_size=32)
correct = sum(r[0]["label"] == t for r, t in zip(results, truths))
print("Accuracy =", correct / len(truths))    

c:\Users\vibha\AppData\Local\Programs\Python\Python312\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Accuracy = 0.8120396039603961
